# 🛒  Customer Purchase Behavior Analyzer

#### 🛠️ Import Libraries

In [129]:
import pandas as pd
import numpy as np
import sqlite3
from scipy import stats
from scipy.stats.mstats import winsorize
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler

##  1️⃣ Data Understanding & Loading

### 📂 Import data from all three sources (CSV, JSON, and SQL).


In [130]:
import pandas as pd
import sqlite3

# 1. Load users data from CSV

users = pd.read_csv("users.csv")

# 2. Load sales data from JSON

sales = pd.read_json("sales.json")

# 3. Load product data from SQL

conn = sqlite3.connect(":memory:")

with open("inventory.sql", "r") as f:
    conn.executescript(f.read())

products = pd.read_sql_query(
    "SELECT * FROM products",
    conn
)

conn.close()

print("Users shape    :", users.shape)
print("Sales shape    :", sales.shape)
print("Products shape :", products.shape)

Users shape    : (200, 6)
Sales shape    : (1000, 6)
Products shape : (50, 5)


🎯 **Insight:** All three sources loaded cleanly — 200 users, 1000 sales transactions, and  50 products — confirming the CSV + JSON + SQL pipeline is working end-to-end and ready for merging.

### 📘 Display top 5 records and data info summary.

In [131]:
print("Users - Top 5 records")
display(users.head())

Users - Top 5 records


,user_id,name,age,gender,city,registration_date
0,U0001,Vihaan Sharma,35,Other,Jaipur,2022-09-08
1,U0002,Sai Reddy,30,Other,Hyderabad,2023-11-24
2,U0003,Aarohi Gupta,37,Other,Indore,2022-02-02
3,U0004,Aarav Gupta,44,Male,Kolkata,2023-06-02
4,U0005,Sara Sharma,30,Other,Chennai,2024-01-04


In [132]:
print("Sales - Top 5 records")
display(sales.head())

Sales - Top 5 records


,transaction_id,user_id,product_id,amount,payment_type,date
0,T000001,U0024,P015,67.67,Wallet,2023-02-12
1,T000002,U0196,P044,76.44,UPI,2023-03-24
2,T000003,U0196,P049,104.57,Debit Card,2025-08-21
3,T000004,U0133,P042,102.75,Net Banking,2024-07-23
4,T000005,U0047,P038,23.89,Net Banking,2025-10-04


In [133]:
print("Products - Top 5 records")
display(products.head())

Products - Top 5 records


,product_id,product_name,category,price,stock
0,P001,Product_001,Grocery,264.89,371
1,P002,Product_002,Grocery,605.91,150
2,P003,Product_003,Beauty,3027.98,127
3,P004,Product_004,Toys,2600.12,229
4,P005,Product_005,Books,1178.99,18


In [134]:
print("----------USERS INFO----------")
users.info()
print("\n----------SALES INFO----------")
sales.info()
print("\n----------PRODUCTS INFO----------")
products.info()

----------USERS INFO----------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   user_id            200 non-null    object
 1   name               200 non-null    object
 2   age                200 non-null    int64 
 3   gender             200 non-null    object
 4   city               200 non-null    object
 5   registration_date  200 non-null    object
dtypes: int64(1), object(5)
memory usage: 9.5+ KB

----------SALES INFO----------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   transaction_id  1000 non-null   object        
 1   user_id         1000 non-null   object        
 2   product_id      1000 non-null   object        
 3   amount          1000 non-null   float

🎯 **Insight:** Dtypes look healthy across the board — sales `amount` is numeric and `date` is already `datetime64`, so no forced type conversions are needed before analysis. 

###  🔍  Identify data types, missing values and inconsistent records

In [135]:
records_before = {
    'users': users.shape[0],
    'sales': sales.shape[0],
    'products': products.shape[0]
}

print("Missing values - users")
print(users.isnull().sum())
print("\nMissing values - sales")
print(sales.isnull().sum())
print("\nMissing values - products")
print(products.isnull().sum())

Missing values - users
user_id              0
name                 0
age                  0
gender               0
city                 0
registration_date    0
dtype: int64

Missing values - sales
transaction_id    0
user_id           0
product_id        0
amount            0
payment_type      0
date              0
dtype: int64

Missing values - products
product_id      0
product_name    0
category        0
price           0
stock           0
dtype: int64


In [136]:
print("\nNegative price entries:", (products['price'] < 0).sum())
print("Negative stock entries:", (products['stock'] < 0).sum())
print("Negative amount entries:", (sales['amount'] < 0).sum())
print("Negative age entries:", (users['age'] < 0).sum())


Negative price entries: 0
Negative stock entries: 0
Negative amount entries: 0
Negative age entries: 0


🎯 **Insight:** Zero missing values and zero negative prices/amounts/ages were found in any of the three raw files — the source data is clean going in, so the imputation steps ahead are precautionary rather than corrective. 

##  2️⃣ Data Cleaning

### 🧮 Handle missing numerical data using SimpleImputer (mean strategy)

In [137]:
num_imputer = SimpleImputer(strategy='mean')

# numerical columns in each dataframe
sales[['amount']] = num_imputer.fit_transform(sales[['amount']])
products[['price', 'stock']] = num_imputer.fit_transform(products[['price', 'stock']])
users[['age']] = num_imputer.fit_transform(users[['age']])

print("Missing numeric values after SimpleImputer:")
print("sales:", sales[['amount']].isnull().sum().sum())
print("products:", products[['price', 'stock']].isnull().sum().sum())
print("users:", users[['age']].isnull().sum().sum())

Missing numeric values after SimpleImputer:
sales: 0
products: 0
users: 0


🎯 **Insight:** SimpleImputer (mean strategy) was applied to `amount`, `price`, `stock`, and `age`; since none of these had missing values, the output confirms the numeric columns were already complete. 0️

###  🗂️ Handle missing categorical data using most-frequent imputation

In [138]:
cat_imputer = SimpleImputer(strategy='most_frequent')

cat_cols_users = ['gender', 'city']
cat_cols_sales = ['payment_type']
cat_cols_products = ['category']

users[cat_cols_users] = cat_imputer.fit_transform(users[cat_cols_users])
sales[cat_cols_sales] = cat_imputer.fit_transform(sales[cat_cols_sales])
products[cat_cols_products] = cat_imputer.fit_transform(products[cat_cols_products])

print("Missing categorical values after imputation:")
print(users[cat_cols_users].isnull().sum())
print(sales[cat_cols_sales].isnull().sum())
print(products[cat_cols_products].isnull().sum())

Missing categorical values after imputation:
gender    0
city      0
dtype: int64
payment_type    0
dtype: int64
category    0
dtype: int64


🎯 **Insight:** Most-frequent imputation was applied to `gender`, `city`, `payment_type`, and `category` — all show zero missing values afterward, meaning the categorical fields didn't need any real fixing. 

### 🔢 Apply KNN Imputer on multivariate data (optional enhancement).


In [139]:
# Demonstrate KNN Imputer on the numeric columns of the sales dataset
knn_imputer = KNNImputer(n_neighbors=5)
sales_numeric = sales[['amount']].copy()
sales_numeric_knn = pd.DataFrame(knn_imputer.fit_transform(sales_numeric), columns=['amount_knn'])
sales['amount'] = sales_numeric_knn['amount_knn']

print("KNN imputation applied on 'amount' column. Sample values:")
display(sales[['amount']].head())

KNN imputation applied on 'amount' column. Sample values:


,amount
0,67.67
1,76.44
2,104.57
3,102.75
4,23.89


🎯 **Insight:** KNN Imputer (k=5) was demonstrated on the `amount` column as the optional enhancement; values stayed identical since there were no gaps, but the technique is now proven ready for messier real-world data. 

### ⚠️ Fix invalid or inconsistent entries (e.g., wrong date formats, negative prices, etc.). 

In [140]:
# Convert date columns to proper datetime format 
sales['date'] = pd.to_datetime(sales['date'], errors='coerce')
users['registration_date'] = pd.to_datetime(users['registration_date'], errors='coerce')

# Fix negative prices,amounts,stock,age if any replace with column mean
for col, df_ in [('price', products), ('stock', products)]:
    df_.loc[df_[col] < 0, col] = np.nan
    df_[col] = df_[col].fillna(df_[col].mean())

sales.loc[sales['amount'] < 0, 'amount'] = np.nan
sales['amount'] = sales['amount'].fillna(sales['amount'].mean())

users.loc[users['age'] < 0, 'age'] = np.nan
users['age'] = users['age'].fillna(users['age'].mean())

In [141]:
users.drop_duplicates(inplace=True)
sales.drop_duplicates(inplace=True)
products.drop_duplicates(inplace=True)

print("Users shape :- ",users.shape)
print("Sales shape :- ",sales.shape)
print("Products shape :- ",products.shape)

Users shape :-  (200, 6)
Sales shape :-  (1000, 6)
Products shape :-  (50, 5)


🎯 **Insight:** Dates were converted to proper datetime format and no negative prices, stock, amounts, or duplicate rows were found — users (200), sales (1000), and products (50) shapes stayed unchanged after cleaning. 

## 3️⃣ Outlier Handling

### 🚫 Detect and remove outliers using Z-score and IQR method.


In [142]:
z_scores = np.abs(stats.zscore(sales['amount']))
outliers_z = (z_scores > 3).sum()
print("Outliers detected in 'amount' using Z-score method:", outliers_z)

Outliers detected in 'amount' using Z-score method: 15


In [143]:
Q1 = sales['amount'].quantile(0.25)
Q3 = sales['amount'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_iqr = ((sales['amount'] < lower_bound) | (sales['amount'] > upper_bound)).sum()
print("Outliers detected in 'amount' using IQR method:", outliers_iqr)

outlier_count_before = int(outliers_iqr)

Outliers detected in 'amount' using IQR method: 53


🎯 **Insight:** Z-score (threshold 3) flagged only 15 outliers in `amount`, while IQR flagged 53 — this gap shows the spend distribution is right-skewed, which pulls fewer points past the strict Z-score cutoff. 

###  📊Compare both techniques and decide which is more suitable for this dataset.


The **IQR method** is more suitable for this dataset because the `amount` column is not perfectly normally distributed (purchase amounts are right-skewed). Z-score assumes a normal distribution, while IQR works well on skewed real-world data, so we proceed with IQR-based treatment.

###  🧰 Apply Winsorization for columns where removal isn’t feasible.

In [144]:
# Winsorize the 'amount' column at 1st and 99th percentile to limit extreme values
sales['amount'] = np.array(winsorize(sales['amount'], limits=[0.01, 0.01]), dtype=float)

# Winsorize product price similarly
products['price'] = np.array(winsorize(products['price'], limits=[0.01, 0.01]), dtype=float)

# recompute outlier count after winsorization
Q1p = sales['amount'].quantile(0.25)
Q3p = sales['amount'].quantile(0.75)
IQRp = Q3p - Q1p
lb, ub = Q1p - 1.5 * IQRp, Q3p + 1.5 * IQRp
outlier_count_after = int(((sales['amount'] < lb) | (sales['amount'] > ub)).sum())

print("Outliers before winsorization:", outlier_count_before)
print("Outliers after winsorization :", outlier_count_after)

Outliers before winsorization: 53
Outliers after winsorization : 53


🎯 **Insight:** Winsorizing `amount` and `price` at the 1st/99th percentile capped extreme values instead of deleting rows, so the IQR outlier count stayed at 53 — record count is preserved while extreme spenders have less pull on averages. 

##  4️⃣ Data Transformation

### 📅 Convert date columns into separate day, month, year features.


In [145]:
sales['sale_day'] = sales['date'].dt.day
sales['sale_month'] = sales['date'].dt.month
sales['sale_year'] = sales['date'].dt.year

users['reg_day'] = users['registration_date'].dt.day
users['reg_month'] = users['registration_date'].dt.month
users['reg_year'] = users['registration_date'].dt.year

display(sales[['date', 'sale_day', 'sale_month', 'sale_year']].head())
display(users[['registration_date', 'reg_day', 'reg_month', 'reg_year']].head())

,date,sale_day,sale_month,sale_year
0,2023-02-12,12,2,2023
1,2023-03-24,24,3,2023
2,2025-08-21,21,8,2025
3,2024-07-23,23,7,2024
4,2025-10-04,4,10,2025


,registration_date,reg_day,reg_month,reg_year
0,2022-09-08,8,9,2022
1,2023-11-24,24,11,2023
2,2022-02-02,2,2,2022
3,2023-06-02,2,6,2023
4,2024-01-04,4,1,2024


🎯 **Insight:** Sale and registration dates were successfully split into day/month/year, unlocking time-based analysis such as seasonality, monthly trends, and customer tenure. 

###  🔠 Encode categorical variables usings :- 

#### 🏷️ Label Encoding for binary columns.


In [146]:
sales['is_wallet_payment'] = (sales['payment_type'] == 'Wallet').astype(int)
le = LabelEncoder()
sales['is_wallet_payment_le'] = le.fit_transform(sales['is_wallet_payment'])
display(sales[['payment_type', 'is_wallet_payment_le']].head())

,payment_type,is_wallet_payment_le
0,Wallet,1
1,UPI,0
2,Debit Card,0
3,Net Banking,0
4,Net Banking,0


🎯 **Insight:** `payment_type` was converted into a binary wallet-usage flag and label-encoded to 0/1, making payment behavior directly usable in numeric models. 

#### 🔤 One-Hot Encoding for nominal columns.


In [147]:
# --- One-Hot Encoding for nominal columns (payment_type, gender) ---
sales = pd.get_dummies(sales, columns=['payment_type'], prefix='pay')
users = pd.get_dummies(users, columns=['gender'], prefix='gender')

display(sales.head())

,transaction_id,user_id,product_id,amount,date,sale_day,sale_month,sale_year,is_wallet_payment,is_wallet_payment_le,pay_Cash,pay_Credit Card,pay_Debit Card,pay_Net Banking,pay_UPI,pay_Wallet
0,T000001,U0024,P015,67.67,2023-02-12,12,2,2023,1,1,False,False,False,False,False,True
1,T000002,U0196,P044,76.44,2023-03-24,24,3,2023,0,0,False,False,False,False,True,False
2,T000003,U0196,P049,104.57,2025-08-21,21,8,2025,0,0,False,False,True,False,False,False
3,T000004,U0133,P042,102.75,2024-07-23,23,7,2024,0,0,False,False,False,True,False,False
4,T000005,U0047,P038,23.89,2025-10-04,4,10,2025,0,0,False,False,False,True,False,False


🎯 **Insight:** One-hot encoding expanded `payment_type` and `gender` into separate binary columns, avoiding any false ordinal relationship between these nominal categories. 

### 📊 Apply binning (e.g., segment customers into spending groups: Low, Medium, High)

In [148]:
user_spend = sales.groupby('user_id')['amount'].sum().reset_index()
user_spend.columns = ['user_id', 'total_spend']

user_spend['spending_group'] = pd.qcut(user_spend['total_spend'], q=3, labels=['Low', 'Medium', 'High'])

# Ordinal encode the spending group (Low < Medium < High)
ordinal_encoder = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])
user_spend['spending_group_encoded'] = ordinal_encoder.fit_transform(user_spend[['spending_group']])

display(user_spend.head())

,user_id,total_spend,spending_group,spending_group_encoded
0,U0001,232.42,Low,0.0
1,U0002,389.04,High,2.0
2,U0003,162.39,Low,0.0
3,U0004,300.64,Medium,1.0
4,U0005,347.69,Medium,1.0


🎯 **Insight:** Customers were split into Low/Medium/High spending tiers using quantile binning and then ordinally encoded — a simple, interpretable segmentation useful for targeting or reporting. 

### 📈 Apply log and square root transformations to normalize skewed data.


In [149]:
sales['amount_log'] = np.log1p(sales['amount'])
sales['amount_sqrt'] = np.sqrt(sales['amount'])

print("Skewness of 'amount'       :", sales['amount'].skew())
print("Skewness of 'amount_log'   :", sales['amount_log'].skew())
print("Skewness of 'amount_sqrt'  :", sales['amount_sqrt'].skew())

Skewness of 'amount'       : 1.4852670734628446
Skewness of 'amount_log'   : 0.06542644671972384
Skewness of 'amount_sqrt'  : 0.7582219779484294


🎯 **Insight:** Raw `amount` skewness was high (≈1.49); the square-root transform brought it down to ≈0.76, and the log transform brought it closest to normal at ≈0.065 — log transformation normalizes this skewed spend data best. 

## 5️⃣ Feature Scaling

### ⚙️  Use StandardScaler and MinMaxScaler to scale numerical features.


In [150]:
num_cols = ['amount']

standard_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

sales['amount_standard_scaled'] = standard_scaler.fit_transform(sales[num_cols])
sales['amount_minmax_scaled'] = minmax_scaler.fit_transform(sales[num_cols])

display(sales[['amount', 'amount_standard_scaled', 'amount_minmax_scaled']].head())

,amount,amount_standard_scaled,amount_minmax_scaled
0,67.67,0.017184,0.259714
1,76.44,0.228679,0.302633
2,104.57,0.907054,0.440296
3,102.75,0.863164,0.431389
4,23.89,-1.038602,0.045463


### 📉 Compare the effect of scaling using summary statistics.


In [151]:
compare_scaling = sales[['amount', 'amount_standard_scaled', 'amount_minmax_scaled']].describe()
display(compare_scaling)

,amount,amount_standard_scaled,amount_minmax_scaled
count,1000.000000,1.000000e+03,1000.000000
mean,66.957430,-4.263256e-17,0.256227
std,41.487469,1.000500e+00,0.203032
min,14.600000,-1.262637e+00,0.000000
25%,37.745000,-7.044789e-01,0.113267
50%,56.390000,-2.548412e-01,0.204512
75%,82.935000,3.853107e-01,0.334418
max,218.940000,3.665170e+00,1.000000


🎯 **Insight:** StandardScaler re-centers `amount` to mean ≈ 0 with unit variance, while MinMaxScaler compresses it into a clean 0–1 range — both keep the original distribution's shape, just on different scales for different model needs. 

## 6️⃣ Feature Construction

#### 🛒 Frequency of purchase.


In [152]:
purchase_freq = sales.groupby('user_id')['transaction_id'].count().reset_index()
purchase_freq.columns = ['user_id', 'purchase_frequency']

#### 💵 Average monthly spend per customer.


In [153]:
avg_spend = sales.groupby('user_id')['amount'].mean().reset_index()
avg_spend.columns = ['user_id', 'avg_spend_per_transaction']

months_active = sales.groupby('user_id')['sale_month'].nunique().reset_index()
months_active.columns = ['user_id', 'active_months']

avg_monthly_spend = user_spend.merge(months_active, on='user_id')
avg_monthly_spend['avg_monthly_spend'] = avg_monthly_spend['total_spend'] / avg_monthly_spend['active_months']

####  ⏱️ Days since last purchase.


In [154]:
reference_date = sales['date'].max()
last_purchase = sales.groupby('user_id')['date'].max().reset_index()
last_purchase.columns = ['user_id', 'last_purchase_date']
last_purchase['days_since_last_purchase'] = (reference_date - last_purchase['last_purchase_date']).dt.days

#### 🖥️ Category-wise total expenditure.


In [155]:
sales_products = sales.merge(products[['product_id', 'category']], on='product_id', how='left')
category_spend = sales_products.groupby(['user_id', 'category'])['amount'].sum().unstack(fill_value=0)
category_spend.columns = [f'spend_{c.lower()}' for c in category_spend.columns]
category_spend = category_spend.reset_index()

In [156]:
print("New engineered features created:")
display(purchase_freq.head())
display(avg_monthly_spend[['user_id', 'avg_monthly_spend']].head())
display(last_purchase[['user_id', 'days_since_last_purchase']].head())
display(category_spend.head())

New engineered features created:


,user_id,purchase_frequency
0,U0001,3
1,U0002,6
2,U0003,4
3,U0004,3
4,U0005,6


,user_id,avg_monthly_spend
0,U0001,77.473333
1,U0002,129.680000
2,U0003,40.597500
3,U0004,100.213333
4,U0005,57.948333


,user_id,days_since_last_purchase
0,U0001,438
1,U0002,553
2,U0003,0
3,U0004,577
4,U0005,132


,user_id,spend_beauty,spend_books,spend_clothing,spend_electronics,spend_grocery,spend_home,spend_sports,spend_toys
0,U0001,42.43,0.00,0.00,44.93,145.06,0.00,0.0,0.00
1,U0002,187.02,0.00,45.02,0.00,0.00,0.00,0.0,157.00
2,U0003,44.45,34.70,0.00,0.00,24.37,58.87,0.0,0.00
3,U0004,222.50,0.00,0.00,0.00,0.00,0.00,0.0,78.14
4,U0005,103.03,68.07,48.34,0.00,0.00,128.25,0.0,0.00


🎯 **Insight:** Four strong customer-level features were engineered — purchase frequency, average monthly spend, days since last purchase, and category-wise spend — together giving an RFM-style view of customer behavior. 

##  7️⃣ Final Dataset Preparation

### 🗃️ Merge all cleaned and engineered datasets

In [157]:
# merge transactional data with user info and product info
final_df = sales_products.merge(users, on='user_id', how='left')

# attach engineered per-customer features
final_df = final_df.merge(purchase_freq, on='user_id', how='left')
final_df = final_df.merge(avg_monthly_spend[['user_id', 'avg_monthly_spend', 'spending_group', 'spending_group_encoded']], on='user_id', how='left')
final_df = final_df.merge(last_purchase[['user_id', 'days_since_last_purchase']], on='user_id', how='left')
final_df = final_df.merge(category_spend, on='user_id', how='left')

print("Final merged dataset shape:", final_df.shape)
display(final_df.head())

Final merged dataset shape: (1000, 44)


,transaction_id,user_id,product_id,amount,date,sale_day,sale_month,sale_year,is_wallet_payment,is_wallet_payment_le,...,spending_group_encoded,days_since_last_purchase,spend_beauty,spend_books,spend_clothing,spend_electronics,spend_grocery,spend_home,spend_sports,spend_toys
0,T000001,U0024,P015,67.67,2023-02-12,12,2,2023,1,1,...,2.0,90,112.61,313.02,89.07,62.27,55.78,46.12,0.0,0.0
1,T000002,U0196,P044,76.44,2023-03-24,24,3,2023,0,0,...,2.0,72,94.93,196.74,0.00,0.00,226.74,104.57,0.0,0.0
2,T000003,U0196,P049,104.57,2025-08-21,21,8,2025,0,0,...,2.0,72,94.93,196.74,0.00,0.00,226.74,104.57,0.0,0.0
3,T000004,U0133,P042,102.75,2024-07-23,23,7,2024,0,0,...,2.0,370,215.07,0.00,0.00,77.16,0.00,158.92,0.0,0.0
4,T000005,U0047,P038,23.89,2025-10-04,4,10,2025,0,0,...,0.0,28,0.00,52.54,0.00,0.00,86.52,23.89,0.0,0.0


🎯 **Insight:** All cleaned and engineered datasets merged smoothly into one final table of 1000 transaction-level rows and 44 columns, tying together demographics, product details, and behavioral features. 

###  📝 Generate a final report showing:
- Number of records before and after cleaning.
- Number of features created.
- Missing value summary (before vs. after).
- Outlier count (before vs. after).

In [158]:
records_after = final_df.shape[0]
features_created = ['sale_day', 'sale_month', 'sale_year', 'reg_day', 'reg_month', 'reg_year',
                     'is_wallet_payment_le', 'amount_log', 'amount_sqrt',
                     'amount_standard_scaled', 'amount_minmax_scaled',
                     'purchase_frequency', 'avg_monthly_spend', 'spending_group',
                     'spending_group_encoded', 'days_since_last_purchase'] + list(category_spend.columns[1:])

missing_after = int(final_df.isnull().sum().sum())

print("==================================================")
print("FINAL DATA PREPROCESSING SUMMARY REPORT")
print("==================================================")
print(f"Records before cleaning (users/sales/products): {records_before}")
print(f"Records in final merged dataset               : {records_after}")
print(f"Number of new features created                 : {len(features_created)}")
print(f"Missing values after cleaning                   : {missing_after}")
print(f"Outlier count before treatment (IQR on amount)  : {outlier_count_before}")
print(f"Outlier count after treatment (IQR on amount)   : {outlier_count_after}")
print("==================================================")

FINAL DATA PREPROCESSING SUMMARY REPORT
Records before cleaning (users/sales/products): {'users': 200, 'sales': 1000, 'products': 50}
Records in final merged dataset               : 1000
Number of new features created                 : 24
Missing values after cleaning                   : 0
Outlier count before treatment (IQR on amount)  : 53
Outlier count after treatment (IQR on amount)   : 53


🎯 **Insight:** The pipeline added 24 new features with zero missing values left, and record count stayed fully intact at 1000 — outlier count held at 53 before and after since winsorization caps values rather than dropping rows. 

## 8️⃣ Bonus (Optional for Extra Credit)

### 🧾 Use Pandas Profiling or YData Profiling to auto-generate an EDA report.

In [159]:
try:
    from ydata_profiling import ProfileReport
    profile = ProfileReport(final_df, title="Customer Purchase Behavior - EDA Report", minimal=True)
    profile.to_file("eda_report.html")
    print("EDA report saved as eda_report.html")
except ImportError:
    print("Install it using: pip install ydata-profiling")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 52.68it/s]

EDA report saved as eda_report.html


### 📊 Save cleaned data as final_cleaned_dataset.csv.

In [160]:
final_df.to_csv('final_cleaned_dataset.csv', index=False)
print("Saved final_cleaned_dataset.csv with shape:", final_df.shape)

Saved final_cleaned_dataset.csv with shape: (1000, 44)


## 🏁 Overall Conclusion

- ✅ The raw data (users, sales, products) was already clean — no missing values, no negatives, no duplicates.
- 📉 IQR proved the better outlier detector for this right-skewed spend data, and winsorization tamed extremes without shrinking the dataset.
- ✨ Log transformation was the most effective fix for skewness in `amount`.
- 🧠 24 new engineered features (RFM-style behavior, category spend, time parts, encodings, scaled values) enrich the dataset well beyond the raw inputs.
- 🚀 The final merged dataset (1000 rows × 44 columns) is fully clean, feature-rich, and ready for downstream analytics or ML modeling.